In [50]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ucphm import *
from functions.new_variables import *
from functions.ensembles import *
from scipy.stats import skew, shapiro

In [51]:
dataset = pd.read_excel('../../BreastCareCond.xlsx')
data_code = pd.read_excel('../../Codificação variaveis.xlsx')

Using the variables she used

This also as the surgery data, the only data that appeared here were this three variables 'Type of 1st intervention', 'Axillary operation', 'SLN procedure'

In [ ]:
#Defining the proposed has good treatment variables to use (the empty ones were removed)
relevant_variables = ['PERSON_ID', 'Regional LN recurrence', 'Date of 1st regional recurrence', 'EBC', 'Survival days']
numerical_variables = ['Age', 'Ki67 positive cells (%) at CB']
categorical_variables = ['N (Regional nodes affected)', 'Number identifying the lesion in the patient', 
                         'Side location of the lesion', 'Classification with respect to other lesions', 
                         'Source of referral', 'Type of invasion at CB', 'Isotype at CB', 
                         'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 
                         'Disease extent', 'Grade at CB', 'Her2 overexpression (with immunohystochemistry) at CB', 'pT']
                        
date_columns = ['Date of CB ', 'Date of 1st breast operation', 'Date of intervention on sentinel lymph node',
                'Date of intervention (axilla)']

In [53]:
#correcting th error in the column name
dataset.rename(columns={'Istotype at CB': 'Isotype at CB'}, inplace=True)

In [54]:
dataset = date_of_diagnosis(dataset, date_columns)

relevant_dates = ['Date of diagnosis', 'Date of 1st regional recurrence']
dataset = remove_zeros_na(dataset, relevant_dates)
dataset = format_date(dataset, relevant_dates)

dataset = survival_days(dataset, ['Date of diagnosis'], ['Date of 1st regional recurrence'])
dataset = tte_negative_patients(dataset)
dataset = filter_EBC(dataset)
dataset.reset_index(drop=True, inplace=True)

In [55]:
#since the clinical staging uses the nodes
node = ['N (Regional nodes affected)']
fill_with_mode(dataset, node)

cT(dataset)
clinical_staging (dataset)
pT(dataset)

# pT was already loaded
new_variables = ['cT', 'Cancer stage']
categorical_variables = categorical_variables + new_variables

In [56]:
change_to_numeric(dataset, numerical_variables)
change_to_categorical(dataset, categorical_variables)

Done
Done


In [57]:
columns_to_map = {
    'Side location of the lesion': {'L': 0, 'R': 1},
    'Source of referral': {1: 0, 2: 1},
    'Type of invasion at CB': {1: 0, 2: 1},
    'Oestrogen receptor status at CB': {1: 0, 2: 1},
    'Progesterone receptor status at CB': {1: 0, 2: 1},
    'Number identifying the lesion in the patient': {1: 1, 2: 2, 3: 3_4, 4: 3_4},
    'pT': {'pTIs': 'pTIs', 'pT1': 'pT1', 'pT2': 'pT2', 'pT3': 'pT3_4', 'pT4': 'pT3_4'},
    'Isotype at CB': {1: 1, 2: 2, 3: 4, 4: 4},
    'N (Regional nodes affected)': {0: 0, 1: 1, 2: 2_3, 3: 2_3}
}

for column, mapping in columns_to_map.items():
    dataset[column] = dataset[column].map(mapping)

In [58]:
# Replace missing values in the specified columns
mode_variables = ['Side location of the lesion', 'Classification with respect to other lesions', 
'Type of invasion at CB', 'Disease extent', 'Oestrogen receptor status at CB', 'pT', 'cT', 'Cancer stage', 'Isotype at CB']
fill_with_mode(dataset, mode_variables)

'Done'

In [59]:
handle_missing = ['Progesterone receptor status at CB', 'Grade at CB', 'Her2 overexpression (with immunohystochemistry) at CB']
dataset, new_columns = one_hot_encode_with_missing_indicator(dataset, handle_missing)
categorical_variables = [col for col in categorical_variables if col not in handle_missing]
categorical_variables.extend(new_columns)

In [60]:
ki67 = ['Ki67 positive cells (%) at CB']
median_imputer(dataset, ki67)

'Done'

In [61]:
one_hot_variables = ['N (Regional nodes affected)', 'Number identifying the lesion in the patient', 'Classification with respect to other lesions', 
                     'Isotype at CB', 'Disease extent', 'cT', 'pT', 'Cancer stage']
ohe_dataset, new_column_names = one_hot_encoding(dataset, one_hot_variables, drop_first=None)

In [62]:
categorical_variables = [col for col in categorical_variables if col not in one_hot_variables]
categorical_variables.extend(new_column_names.tolist())

selected_variables = relevant_variables + numerical_variables + categorical_variables

In [63]:
treatment_variables = ['Radiotherapy (RT) performed', 'Conventional RT fraction', 'Total administered dose', 'Boost dose administered', 
                       'Radiotherapy on chest wall', 'Radiotherapy on supraclavicular area', 'Radiotherapy on internal mammary chain',
                         'State (if hormone therapy prescribed)', 'Treatment in association with chemotherapy', 
                        'Chemotherapy performed', 'Adjuvant chemotherapy', 'Adjuvant therapy with biological drugs', 'Endocrine therapy performed']

dataset = ohe_dataset

In [64]:
numerical_data, categorical_data = type_separation(data_code, treatment_variables)
len(numerical_data), len(categorical_data)

(3, 10)

In [65]:
replace_missing_values(dataset, numerical_data, 0)

Done


In [66]:
dataset.drop(index=0, inplace=True)
dataset.reset_index(inplace=True, drop=True)
replace_missing_values(dataset, categorical_data, 0)

Done


In [67]:
variable_two_to_zero = ['Adjuvant chemotherapy', 'Chemotherapy performed', 'Endocrine therapy performed']
variable_two_to_one = ['Adjuvant therapy with biological drugs']

two_to_zero(dataset, variable_two_to_zero)
two_to_one(dataset, variable_two_to_one)

'Done'

In [74]:
to_remove = ['PERSON_ID', 'Regional LN recurrence', 'Date of 1st regional recurrence', 'EBC', 'Survival days']
important_variables = ['PERSON_ID', 'Regional LN recurrence', 'Survival days']
diagnosis_variables = [col for col in selected_variables if col not in to_remove]
selected_columns = important_variables + treatment_variables + diagnosis_variables

In [75]:
survival_dataset = dataset[selected_columns]

In [70]:
# Check if there are any NaN values in the dataset
contains_nan = survival_dataset.isnull().any().any()
print("Does survival_dataset contain NaN values?", contains_nan)

# Optionally, to see which columns contain NaN values
nan_columns = survival_dataset.columns[survival_dataset.isnull().any()].tolist()
print("Columns with NaN values:", nan_columns)

Does survival_dataset contain NaN values? False
Columns with NaN values: []


In [22]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False) & (survival_dataset['Regional LN recurrence'] == 1))

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False) & (survival_dataset['Regional LN recurrence'] == 0))

# For each duplicated PERSON_ID, keep only the row with the highest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, False])
    .groupby('PERSON_ID')
    .tail(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [23]:
survival_dataset.to_csv("data.csv")

In [24]:
with open("diagnosis.txt", "w") as file:
    for item in diagnosis_variables:
        file.write(str(item) + "\n")
with open("treatment.txt", "w") as file:
    for item in treatment_variables:
        file.write(str(item) + "\n")